# ⚖️ RQ6 — Regimes de treino: consolidar ou preservar a divergência?

Compara os quatro regimes do RoBERTa (Seção 4.6) **pareados por fold**, com o mesmo método da
comparação de sistemas de linha de base (`comparacao_sistemas_finetuning.ipynb`):

1. **IC 95% individual** de cada regime (t de Student);
2. **IC 95% da diferença pareada** por fold — contém 0 ⇒ equivalentes;
3. **t pareado** com correção de Bonferroni e Benjamini-Hochberg (dentro de cada métrica).

| Regime | `training_mode` |
|---|---|
| Voto majoritário | `aggregated` |
| Majoritário ×3 (controle da repetição) | `aggregated_replicated` |
| Perspectivista | `perspectivism` |
| Soft labels | `soft_labels` |

Métricas: macro-F1 (inclusive contra o **benchmark**), ECE e Brier — no conjunto de teste inteiro e
separados pelos estratos de concordância das LLMs **3x0** e **2x1**. Em ECE/Brier, menor é melhor.

## 1) Setup e configuração

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
while not (ROOT / "data" / "results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.systems.fine_tune_system.evaluation.regime_comparison import RegimeComparison

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

CONF = 0.95
RESULTS_DIR = ROOT / "data" / "results"
XLSX_BENCHMARK = RESULTS_DIR / "mq" / "Roberta - Catizani.xlsx"
EXPORT_DIR = RESULTS_DIR / "mq" / "comparacao_regimes"

# Pasta da anotação (última rodada) de cada dataset — fixe a data para não depender do "latest" por mtime
ANNOTATION_DATES = {
    "movie_review": "latest",
    "books": "2026-09-13_07-19-54",
    "dblp": "latest",
}

NOMES = {
    "aggregated": "Majoritário",
    "aggregated_replicated": "Majoritário ×3",
    "perspectivism": "Perspectivista",
    "soft_labels": "Soft labels",
    "benchmark": "Benchmark",
}

## 2) Execuções usadas e métricas por fold

In [ ]:
comparison = RegimeComparison(RESULTS_DIR, ANNOTATION_DATES, benchmark_xlsx=XLSX_BENCHMARK, conf=CONF)
for ds in ANNOTATION_DATES:
    print(ds, "→", comparison.annotation_dir(ds).name)
    for regime, run_dir in comparison.locate_runs(ds).items():
        print(f"    {NOMES[regime]:<16} {run_dir.name}")

tables = comparison.run()
folds, individual, paired = tables["folds"], tables["individual"], tables["paired"]
folds[folds["metric"] == "f1_macro"].pivot_table(index=["dataset", "fold"], columns="system", values="value")

## 3) IC 95% individual — macro-F1

In [ ]:
ic_f1 = individual[individual["metric"] == "f1_macro"].copy()
ic_f1["system"] = ic_f1["system"].map(NOMES)
ic_f1

## 4) Diferença pareada — macro-F1 (cada regime vs benchmark e contrastes entre regimes)

In [ ]:
COLS = ["dataset", "system_a", "system_b", "n_folds", "mean_a", "mean_b", "diff_mean",
        "diff_ci_low", "diff_ci_high", "conclusion", "t_paired_p", "p_bonferroni", "p_bh"]
paired[paired["metric"] == "f1_macro"][COLS]

## 5) Calibração — ECE e Brier (geral e por estrato 3x0 / 2x1)

In [ ]:
for metric in ["ece", "ece_3x0", "ece_2x1", "brier", "brier_3x0", "brier_2x1"]:
    print(f"\n=== {metric} (diff = A − B; negativo ⇒ A melhor) ===")
    display(paired[paired["metric"] == metric][COLS])

## 6) macro-F1 por estrato de concordância

In [ ]:
for metric in ["f1_macro_3x0", "f1_macro_2x1"]:
    print(f"\n=== {metric} ===")
    display(paired[paired["metric"] == metric][COLS])

## 7) Gráficos — IC 95% da diferença (forest plot)

In [ ]:
def forest(metric, title):
    d = paired[paired["metric"] == metric].reset_index(drop=True)
    labels = d["dataset"] + ": " + d["system_a"].map(NOMES) + " − " + d["system_b"].map(NOMES)
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(d) + 1.5))
    y = np.arange(len(d))[::-1]
    for yi, (_, r) in zip(y, d.iterrows()):
        cor = "#888" if r["diff_ci_low"] <= 0 <= r["diff_ci_high"] else "#C8102E"
        ax.plot([r["diff_ci_low"], r["diff_ci_high"]], [yi, yi], "-", color=cor, lw=3)
        ax.plot(r["diff_mean"], yi, "o", color=cor, ms=7)
    ax.axvline(0, color="black", ls="--", lw=1)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("diferença (A − B), IC 95%"); ax.set_title(title)
    plt.tight_layout(); plt.show()


forest("f1_macro", "macro-F1 — IC 95% da diferença pareada (cinza ⇒ equivalentes)")
forest("ece_2x1", "ECE nos casos 2x1 — negativo ⇒ A mais bem calibrado")
forest("ece_3x0", "ECE nos casos 3x0 — negativo ⇒ A mais bem calibrado")

## 8) Exportar — `data/results/mq/comparacao_regimes`

In [ ]:
path = comparison.export(tables, EXPORT_DIR)
print("✓ Salvo em:", EXPORT_DIR)
for p in sorted(EXPORT_DIR.glob("*")):
    print("  -", p.name)